# Replicating RefractiveIndex.INFO with `refractivesqlite`

This notebook demonstrates how to reproduce the information shown on the
[refractiveindex.info](https://refractiveindex.info) website using the
`refractivesqlite` Python package.

We replicate the page for **SiO₂ (Silicon dioxide, Silica, Quartz) — Malitson 1965**,
which covers the refractive index from 0.21 to 6.7 µm using a 3-term Sellmeier formula.

Sections:
1. Load the material
2. Refractive index at a specific wavelength
3. n(λ) and k(λ) plots
4. Derived optical constants (Abbe number, chromatic dispersion, group index, GVD)
5. Dispersion formula and coefficients
6. Complex permittivity ε
7. Fresnel reflection calculator

---
## 1. Load the material

In [ ]:
from refractivesqlite import Database

db = Database("refractive.db")

# Search for SiO2 Malitson
db.search_pages("Malitson")

In [ ]:
# Load the material — adjust pageid based on your database
# Typically SiO2 Malitson is one of the first entries.
# Use the pageid from the search above.
import numpy as np

# Find SiO2 Malitson programmatically
rows = db.search_custom(
    "SELECT pageid FROM pages WHERE book LIKE '%SiO2%' AND page LIKE '%Malitson%'"
)
PAGEID = rows[0][0] if rows else None
print(f"SiO2 Malitson pageid = {PAGEID}")

mat = db.get_material(PAGEID)
info = mat.get_page_info()
print(f"Shelf: {info['shelf']}")
print(f"Book:  {info['book']}")
print(f"Page:  {info['page']}")
print(f"Range: {info['rangeMin']} – {info['rangeMax']} µm")
print(f"Has n: {mat.has_refractive()}, Has k: {mat.has_extinction()}")

---
## 2. Refractive index at a specific wavelength

The website shows *n* = 1.4585 at λ = 0.5876 µm (the sodium d-line).

In [ ]:
# Sodium d-line: 587.5618 nm = 0.5876 µm
wl_d = 587.5618  # nm

n_d = mat.get_refractiveindex(wl_d)
print(f"n at {wl_d} nm = {n_d:.4f}")
print(f"n at 0.5876 µm = {mat.get_refractiveindex(0.5876, unit='um'):.4f}")

# Compare with website value
print(f"\nWebsite shows: n = 1.4585")

---
## 3. n(λ) plot

Reproduce the n vs wavelength plot shown on the website.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get wavelength range
wl_min, wl_max = mat.get_wl_range(unit='um')
print(f"Valid range: {wl_min:.2f} – {wl_max:.1f} µm")

# Dense wavelength grid in µm
wl_um = np.linspace(wl_min, wl_max, 500)
n_values = mat.get_refractiveindex(wl_um, unit='um')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(wl_um, n_values, color='purple', linewidth=2)
ax.set_xlabel('Wavelength, µm', fontsize=12)
ax.set_ylabel('n', fontsize=12)
ax.set_title(
    f"RefractiveIndex.INFO\n"
    f"{info['book']}\n"
    f"{info['page']} n {wl_min:.2f}–{wl_max:.1f} µm",
    fontsize=11
)
ax.grid(True, alpha=0.3)

# Mark the d-line
ax.axvline(0.5876, color='gray', linestyle='--', alpha=0.5, label='d-line (587.6 nm)')
ax.plot(0.5876, n_d, 'ro', markersize=6, label=f'n = {n_d:.4f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 4. Derived optical constants

The website computes several derived quantities from the dispersion curve.
We compute them here using numerical differentiation.

### Standard spectral lines

| Line | Symbol | Wavelength (nm) |
|------|--------|-----------------|
| Hydrogen C | C | 656.2725 |
| Sodium d | d | 587.5618 |
| Hydrogen F | F | 486.1327 |

In [ ]:
import numpy as np

# Standard spectral lines (nm)
wl_C = 656.2725   # H-alpha (red)
wl_d = 587.5618   # Na d-line (yellow)
wl_F = 486.1327   # H-beta (blue)

n_C = mat.get_refractiveindex(wl_C)
n_d = mat.get_refractiveindex(wl_d)
n_F = mat.get_refractiveindex(wl_F)

print(f"n_C ({wl_C} nm) = {n_C:.6f}")
print(f"n_d ({wl_d} nm) = {n_d:.6f}")
print(f"n_F ({wl_F} nm) = {n_F:.6f}")

### 4.1 Abbe number

$$V_d = \frac{n_d - 1}{n_F - n_C}$$

The website shows $V_d = 67.82$.

In [ ]:
V_d = (n_d - 1) / (n_F - n_C)

print(f"Abbe number V_d = {V_d:.2f}")
print(f"Website value:    67.82")

### 4.2 Chromatic dispersion dn/dλ

Numerical derivative at the d-line. The website shows dn/dλ = −0.035209 µm⁻¹.

In [ ]:
import numpy as np

def numerical_derivative(mat, wl_nm, order=1, delta_nm=0.01, unit='nm'):
    """Central-difference numerical derivative of n(λ).
    Returns dn/dλ in units of 1/[unit].
    """
    if order == 1:
        n_plus  = mat.get_refractiveindex(wl_nm + delta_nm, unit=unit)
        n_minus = mat.get_refractiveindex(wl_nm - delta_nm, unit=unit)
        return (n_plus - n_minus) / (2 * delta_nm)
    elif order == 2:
        n_plus  = mat.get_refractiveindex(wl_nm + delta_nm, unit=unit)
        n_0     = mat.get_refractiveindex(wl_nm, unit=unit)
        n_minus = mat.get_refractiveindex(wl_nm - delta_nm, unit=unit)
        return (n_plus - 2 * n_0 + n_minus) / (delta_nm ** 2)

# dn/dλ at d-line, in µm⁻¹
wl_d_um = 0.5876
dndl = numerical_derivative(mat, wl_d_um, order=1, delta_nm=0.0001, unit='um')

print(f"dn/dλ at d-line = {dndl:.6f} µm⁻¹")
print(f"Website value:    -0.035209 µm⁻¹")

### 4.3 Group index

$$n_g = n - \lambda \frac{dn}{d\lambda}$$

The website shows $n_g = 1.4792$.

In [ ]:
n_g = n_d - wl_d_um * dndl

print(f"Group index n_g = {n_g:.4f}")
print(f"Website value:    1.4792")

### 4.4 Group velocity dispersion (GVD)

$$\text{GVD} = \frac{\lambda^3}{2\pi c^2} \frac{d^2 n}{d\lambda^2}$$

in fs²/mm. The website shows GVD = 57.549 fs²/mm.

The dispersion parameter $D$ (used in fiber optics) is:

$$D = -\frac{\lambda}{c} \frac{d^2 n}{d\lambda^2}$$

in ps/(nm·km). The website shows $D = -313.91$ ps/(nm·km).

In [ ]:
import numpy as np

c = 299792458.0  # m/s

# d²n/dλ² at d-line (λ in µm)
d2ndl2 = numerical_derivative(mat, wl_d_um, order=2, delta_nm=0.0001, unit='um')
print(f"d²n/dλ² = {d2ndl2:.4f} µm⁻²")

# Convert to SI for GVD: λ in m, dn/dλ² in m⁻²
wl_m = wl_d_um * 1e-6  # µm → m
d2ndl2_m = d2ndl2 * 1e12  # µm⁻² → m⁻²

# GVD = λ³/(2πc²) × d²n/dλ² [in s²/m]
GVD_s2_per_m = (wl_m**3 / (2 * np.pi * c**2)) * d2ndl2_m

# Convert to fs²/mm
GVD_fs2_per_mm = GVD_s2_per_m * 1e30 * 1e-3  # s²→fs² (*1e30), m→mm (*1e-3)
print(f"\nGVD = {GVD_fs2_per_mm:.3f} fs²/mm")
print(f"Website value: 57.549 fs²/mm")

# D parameter = -λ/c × d²n/dλ² [in s/m²]
D_s_per_m2 = -(wl_m / c) * d2ndl2_m

# Convert to ps/(nm·km)
D_ps_per_nm_km = D_s_per_m2 * 1e12 * 1e-9 * 1e3  # s→ps, m→nm (÷), m→km
print(f"\nD = {D_ps_per_nm_km:.2f} ps/(nm·km)")
print(f"Website value: -313.91 ps/(nm·km)")

---
## 5. Dispersion formula and coefficients

SiO₂ Malitson uses a 3-term Sellmeier formula:

$$n^2 - 1 = \frac{B_1 \lambda^2}{\lambda^2 - C_1^2} + \frac{B_2 \lambda^2}{\lambda^2 - C_2^2} + \frac{B_3 \lambda^2}{\lambda^2 - C_3^2}$$

The website shows:
- $B_1$ = 0.6961663, $C_1²$ = 0.0684043²
- $B_2$ = 0.4079426, $C_2²$ = 0.1162414²
- $B_3$ = 0.8974794, $C_3²$ = 9.896161²

**Note:** `db.get_material()` loads tabulated data from SQLite — the
original formula coefficients are not stored there.  To access the
dispersion formula, use `db.get_material_from_yml()` which reads the
original YAML file directly.  This requires the YML database folder
(downloaded during `create_database_from_url`).

In [ ]:
# Load material from YAML to access the original formula
# Pass the path to the YML database folder (contains catalog-nk.yml or library.yml)
mat_yml = db.get_material_from_yml(PAGEID, yml_database_path="database")

if mat_yml is None:
    print("YAML database folder not found. Skipping formula display.")
    print("To use this cell, ensure the 'database/' folder exists")
    print("(it is created by db.create_database_from_url()).")
else:
    from refractivesqlite.optical_data import FormulaRefractiveIndexData

    ri = mat_yml.refractiveIndex
    if isinstance(ri, FormulaRefractiveIndexData):
        FORMULA_NAMES = {
            1: "Sellmeier", 2: "Sellmeier-2", 3: "Polynomial",
            4: "RefractiveIndex.INFO", 5: "Cauchy", 6: "Gases",
            7: "Herzberger", 8: "Retro", 9: "Exotic",
        }
        print(f"Formula type: {ri.formula}  ({FORMULA_NAMES.get(ri.formula, '?')})")
        print(f"Range: {ri.rangeMin} – {ri.rangeMax} µm")
        print(f"\nCoefficients:")

        C = ri.coefficients
        for i, c in enumerate(C):
            print(f"  C[{i}] = {c}")

        if ri.formula == 1 and len(C) >= 7:
            print(f"\nSellmeier interpretation:")
            print(f"  Constant term (should be 0): {C[0]}")
            for term in range(3):
                B = C[1 + 2*term]
                Csq = C[2 + 2*term]
                print(f"  Term {term+1}: B = {B}, C = {Csq},  C² = {Csq**2:.10f}")

            print(f"\n  n² − 1 = "
                  f"{C[1]}λ²/(λ² − {C[2]}²) + "
                  f"{C[3]}λ²/(λ² − {C[4]}²) + "
                  f"{C[5]}λ²/(λ² − {C[6]}²)")
    else:
        print("This material uses tabulated data, not a formula.")
        print(f"Range: {ri.rangeMin} – {ri.rangeMax} µm")
        print(f"Data points: {len(ri.wavelengths)}")

---
## 6. Complex permittivity ε

The complex dielectric function ε = ε₁ + iε₂ = (n + ik)².

For SiO₂ Malitson (no k data), ε is purely real: ε = n².

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Scalar query
eps_d = mat.get_epsilon(wl_d)
print(f"ε at {wl_d} nm:")
print(f"  ε     = {eps_d}")
print(f"  ε₁    = {eps_d.real:.6f}  (= n² = {n_d**2:.6f})")
print(f"  ε₂    = {eps_d.imag:.6f}  (= 0, no k data)")

# Array query
wl_um = np.linspace(0.21, 6.7, 500)
eps = mat.get_epsilon(wl_um, unit='um')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(wl_um, eps.real, color='steelblue', linewidth=2, label=r'$\varepsilon_1$ (= n²)')
if np.any(eps.imag != 0):
    ax.plot(wl_um, eps.imag, color='tomato', linewidth=2, label=r'$\varepsilon_2$ (= 2nk)')
ax.set_xlabel('Wavelength (µm)', fontsize=12)
ax.set_ylabel(r'$\varepsilon$', fontsize=14)
ax.set_title('SiO₂ Malitson — Dielectric function', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.1 Permittivity for a metal (with k)

Let's also show ε for a material that *has* extinction data, to demonstrate both ε₁ and ε₂.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Find a metal with both n and k — e.g., Au or Ag
rows = db.search_custom(
    "SELECT pageid, page FROM pages WHERE book='Au' AND hasrefractive=1 AND hasextinction=1 LIMIT 1"
)
if rows:
    au_id, au_page = rows[0]
    mat_au = db.get_material(au_id)

    lo, hi = mat_au.get_wl_range(unit='nm')
    wls = np.linspace(lo, hi, 300)
    eps_au = mat_au.get_epsilon(wls)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(wls, eps_au.real, color='steelblue', linewidth=2)
    ax1.axhline(0, color='k', linewidth=0.6, linestyle='--')
    ax1.set_xlabel('Wavelength (nm)')
    ax1.set_ylabel(r'$\varepsilon_1$  (n² − k²)')
    ax1.set_title(f'Au / {au_page} — Re(ε)', fontsize=11)
    ax1.grid(True, alpha=0.3)

    ax2.plot(wls, eps_au.imag, color='tomato', linewidth=2)
    ax2.set_xlabel('Wavelength (nm)')
    ax2.set_ylabel(r'$\varepsilon_2$  (2nk)')
    ax2.set_title(f'Au / {au_page} — Im(ε)', fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No Au data with both n and k found in the database.")

---
## 7. Fresnel reflection calculator

Reproduce the Fresnel reflection calculator from the website.

At normal incidence (θ = 0°) from air (n₁ = 1) into the material (n₂ = n):

$$R = \left(\frac{n_1 - n_2}{n_1 + n_2}\right)^2$$

For arbitrary angle, the Fresnel equations give $R_p$ and $R_s$ separately.

The website shows at λ = 0.5876 µm, θ = 0°:
- $R_p = R_s = R = 0.034776$
- Brewster's angle = 55.563°

In [ ]:
import numpy as np

n1 = 1.0      # air
n2 = n_d       # SiO2 at d-line

# Normal incidence
R = ((n1 - n2) / (n1 + n2))**2
print(f"At λ = {wl_d} nm, normal incidence (θ = 0°):")
print(f"  n₂ = {n2:.4f}")
print(f"  R_p = R_s = R = {R:.6f}")
print(f"  Website value:    0.034776")

# Brewster's angle
theta_B = np.degrees(np.arctan(n2 / n1))
print(f"\n  Brewster's angle θ_B = {theta_B:.3f}°")
print(f"  Website value:         55.563°")

In [ ]:
import numpy as np

def fresnel_reflectance(n1, n2, theta_i_deg):
    """Compute Fresnel reflectances R_s, R_p for an interface.

    Parameters
    ----------
    n1, n2 : float
        Refractive indices of medium 1 (incident) and medium 2.
    theta_i_deg : float or array-like
        Angle of incidence in degrees.

    Returns
    -------
    R_s, R_p : float or ndarray
    """
    theta_i = np.radians(np.asarray(theta_i_deg, dtype=float))
    cos_i = np.cos(theta_i)
    sin_i = np.sin(theta_i)

    # Snell's law: n1 sin(θ_i) = n2 sin(θ_t)
    sin_t = n1 / n2 * sin_i
    cos_t = np.sqrt(1 - sin_t**2 + 0j)  # complex for TIR

    # Fresnel coefficients (amplitude)
    r_s = (n1 * cos_i - n2 * cos_t) / (n1 * cos_i + n2 * cos_t)
    r_p = (n2 * cos_i - n1 * cos_t) / (n2 * cos_i + n1 * cos_t)

    R_s = np.abs(r_s)**2
    R_p = np.abs(r_p)**2

    return R_s, R_p


def fresnel_phase(n1, n2, theta_i_deg):
    """Compute Fresnel reflection phase shifts."""
    theta_i = np.radians(np.asarray(theta_i_deg, dtype=float))
    cos_i = np.cos(theta_i)
    sin_i = np.sin(theta_i)
    sin_t = n1 / n2 * sin_i
    cos_t = np.sqrt(1 - sin_t**2 + 0j)

    r_s = (n1 * cos_i - n2 * cos_t) / (n1 * cos_i + n2 * cos_t)
    r_p = (n2 * cos_i - n1 * cos_t) / (n2 * cos_i + n1 * cos_t)

    phi_s = np.degrees(np.angle(r_s))
    phi_p = np.degrees(np.angle(r_p))

    return phi_s, phi_p

print("Fresnel functions defined.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

theta = np.linspace(0, 90, 500)
R_s, R_p = fresnel_reflectance(n1, n2, theta)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(theta, R_p, color='steelblue', linewidth=2, label=r'$R_p$')
ax.plot(theta, R_s, color='tomato', linewidth=2, label=r'$R_s$')
ax.plot(theta, (R_s + R_p) / 2, color='green', linewidth=1.5,
        linestyle='--', label=r'$R = (R_p + R_s)/2$')

# Mark Brewster's angle
ax.axvline(theta_B, color='gray', linestyle=':', alpha=0.7)
ax.annotate(f"θ_B = {theta_B:.1f}°", xy=(theta_B, 0.02),
            fontsize=10, ha='left', va='bottom')

ax.set_xlabel('Angle of incidence (degrees)', fontsize=12)
ax.set_ylabel('Reflectance', fontsize=12)
ax.set_title(
    f'Fresnel reflection — SiO₂ at {wl_d} nm (n = {n2:.4f})',
    fontsize=12
)
ax.set_xlim(0, 90)
ax.set_ylim(0, 1)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

theta = np.linspace(0, 90, 500)
phi_s, phi_p = fresnel_phase(n1, n2, theta)

# At normal incidence, website shows φ_p = 0°, φ_s = 180°
print(f"Reflection phase at θ = 0°:")
phi_s_0, phi_p_0 = fresnel_phase(n1, n2, 0)
print(f"  φ_p = {float(phi_p_0.real):.0f}°    (website: 0°)")
print(f"  φ_s = {float(phi_s_0.real):.0f}°  (website: 180°)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta, phi_p.real, color='steelblue', linewidth=2, label=r'$\phi_p$')
ax.plot(theta, phi_s.real, color='tomato', linewidth=2, label=r'$\phi_s$')
ax.axvline(theta_B, color='gray', linestyle=':', alpha=0.7,
           label=f'Brewster ({theta_B:.1f}°)')
ax.set_xlabel('Angle of incidence (degrees)', fontsize=12)
ax.set_ylabel('Reflection phase (degrees)', fontsize=12)
ax.set_title('Fresnel reflection phase — SiO₂', fontsize=12)
ax.set_xlim(0, 90)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Multi-dataset comparison

The website also lets you compare multiple datasets for the same material.
Let's compare all SiO₂ refractive index sources.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Find all SiO2 pages with refractive index
rows = db.search_custom(
    "SELECT pageid, page FROM pages WHERE book LIKE '%SiO2%' AND hasrefractive=1"
)

fig, ax = plt.subplots(figsize=(10, 6))

for pageid, page_name in rows:
    m = db.get_material(pageid)
    lo, hi = m.get_wl_range(unit='um')
    # Only plot if range overlaps visible/near-IR
    if hi > 0.2 and lo < 5:
        wl = np.linspace(max(lo, 0.2), min(hi, 5), 200)
        n = m.get_refractiveindex(wl, unit='um')
        ax.plot(wl, n, linewidth=1.5, label=page_name)

ax.set_xlabel('Wavelength (µm)', fontsize=12)
ax.set_ylabel('Refractive index n', fontsize=12)
ax.set_title('SiO₂ — All available datasets', fontsize=12)
ax.set_xlim(0.2, 5)
ax.legend(fontsize=7, ncol=2, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary of website features replicated

| Website feature | Package method / code |
|---|---|
| Material lookup | `db.search_pages()`, `db.get_material(pageid)` |
| n at a wavelength | `mat.get_refractiveindex(λ, unit=)` |
| k at a wavelength | `mat.get_extinctioncoefficient(λ, unit=)` |
| n(λ) / k(λ) plot | `mat.get_refractiveindex(array, unit=)` + matplotlib |
| Wavelength range | `mat.get_wl_range(unit=)` |
| Complex permittivity ε | `mat.get_epsilon(λ, unit=)` → `.real`, `.imag` |
| Abbe number | `(n_d − 1) / (n_F − n_C)` |
| Chromatic dispersion | Numerical `dn/dλ` |
| Group index | `n − λ dn/dλ` |
| GVD, D parameter | `λ³/(2πc²) d²n/dλ²` |
| Dispersion formula | `db.get_material_from_yml()` → `ri.formula`, `ri.coefficients` |
| Fresnel R_p, R_s, R | `fresnel_reflectance(n1, n2, θ)` |
| Reflection phase | `fresnel_phase(n1, n2, θ)` |
| Brewster's angle | `arctan(n2/n1)` |
| Multi-dataset comparison | Loop over `search_custom()` results |